Credit Risk Predictions w/ Macro Shocks \
Datasets from LendingClub Loan (borrower demographics, loan terms, repayment and default), FRED Macroeconomic Data (unemployment rate, GDP growth, Federal Reserve fund rates, credit spreads) \
Research Question: Do macroeconomic shocks improve machine learning models of household credit risk beyond borrower-level characteristics?

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import seaborn as sns

from scipy import stats

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, log_loss, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, classification_report

In [2]:
import kagglehub
path = kagglehub.dataset_download("adarshsng/lending-club-loan-data-csv")
print("Path to dataset files:", path)

100%|██████████| 339M/339M [00:04<00:00, 72.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/adarshsng/lending-club-loan-data-csv/versions/1


In [6]:
df = pd.read_csv("/kaggle/input/lending-club-loan-data-csv/loan.csv")
df
print(df.columns.tolist())

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/lending-club-loan-data-csv/loan.csv'

In [ ]:
cols = ["loan_amnt", "term", "int_rate", "grade", "sub_grade", "emp_length", "home_ownership", "annual_inc", "dti", "purpose", "addr_state", "issue_d", "loan_status"]
df = df[cols]

In [ ]:
df = df[df["loan_status"].isin(["Charged Off", "Fully Paid"])]
df["default"] = (df["loan_status"] == "Charged Off").astype(int)

In [ ]:
unemp = web.DataReader("UNRATE", "fred")      # unemployment rate
fedfunds = web.DataReader("FEDFUNDS", "fred") # fed funds rate

In [ ]:
macro = pd.concat([unemp, fedfunds], axis=1)
macro.columns = ["unemp","fedfunds"]
macro.index = pd.to_datetime(macro.index)

In [ ]:
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df = df.merge(macro, left_on="issue_d", right_index=True, how="left")

In [ ]:
X = df.drop(columns=["loan_status", "default"])
y = df["default"]

categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=np.number).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')), ('passthrough', 'passthrough')]), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
reg = LinearRegression()
reg.fit(X_train_processed, y_train)

slope = reg.coef_
intercept = reg.intercept_
print(f"Slope: {slope}")
print(f"Intercept: {intercept}")

y_pred = reg.predict(X_test_processed)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mean Squared Error: {mse}")
print(f"R-squared: {r2}")

feature_names = preprocessor.get_feature_names_out()
coef_table = pd.DataFrame({
    "feature": feature_names,
    "slope": reg.coef_
})
print(coef_table.sort_values("slope", key=abs, ascending=False).head(15))

In [ ]:
logit = LogisticRegression(max_iter=2000, n_jobs=-1)
logit.fit(X_train_processed, y_train)

print("Intercept (log-odds):", logit.intercept_[0])

y_proba = logit.predict_proba(X_test_processed)[:, 1]
y_hat = (y_proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Log Loss:", log_loss(y_test, y_proba))
print("Accuracy (0.5 cutoff):", accuracy_score(y_test, y_hat))

p = logit.predict_proba(X_test_processed)[:, 1]
avg_deriv_factor = np.mean(p * (1 - p))

betas = logit.coef_.ravel()
feature_names = preprocessor.get_feature_names_out()
marginal_effects = betas * avg_deriv_factor

me_table = pd.DataFrame({
    "feature": feature_names,
    "marginal_effect": marginal_effects
}).sort_values("marginal_effect", key=abs, ascending=False)

print("\n=== Average Marginal Effects (Top 15) ===")
print(me_table.head(15))

In [ ]:
sns.histplot(data=df['default'], discrete=True)
plt.title("Loan Default Distribution")
plt.xlabel("Default (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

In [ ]:
# people sit below $200k annual income (most borrowers are average income)
# both defaulters and nondefaulters show similar incomes
# some making $500k default
# income is not a strong predicter of default
df["income_thousands"] = df["annual_inc"] / 1000

plt.figure(figsize=(8,6))
sns.stripplot(
    data=df.sample(5000, random_state=42),
    x="default",
    y="income_thousands",
    alpha=0.3,
    jitter=True
)

plt.title("Annual Income vs Loan Default")
plt.xlabel("Default (0 = No, 1 = Yes)")
plt.ylabel("Annual Income (Thousands $)")
plt.show()

In [ ]:
print("Classification Report:\n", classification_report(y_test, y_hat))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

fpr, tpr, thresholds = roc_curve(y_test, y_proba)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_proba):.3f}")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()